# Imports

In [1]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine, text

# database connections

In [2]:
with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)

config_co = config["CO_SA"]
config_etl = config["ETL_PRO"]

co_sa = create_engine(
    f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:{config_co['port']}/{config_co['dbname']}"
)
etl_conn = create_engine(
    f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:{config_etl['port']}/{config_etl['dbname']}"
)

# Extract

In [3]:
df_tipo = pd.read_sql_table("mensajeria_tiponovedad", con=co_sa)
df_tipo

,id,nombre
0,2,No puedo continuar
1,1,Novedades del servicio


# Transformations

In [4]:
dim_novedad = df_tipo[['id', 'nombre']].rename(columns={
    'id': 'id_novedad',
    'nombre': 'categoria_novedad'
}).copy()

dim_novedad = dim_novedad.sort_values('id_novedad').reset_index(drop=True)
dim_novedad.insert(0, 'key_dim_novedad', range(1, len(dim_novedad) + 1))
dim_novedad

,key_dim_novedad,id_novedad,categoria_novedad
0,1,1,Novedades del servicio
1,2,2,No puedo continuar


# Create table

In [5]:
sql = """
DROP TABLE IF EXISTS dim_novedad;

CREATE TABLE dim_novedad(
    key_dim_novedad   INTEGER PRIMARY KEY,
    id_novedad        INTEGER NOT NULL,
    categoria_novedad VARCHAR(150)
);
"""

with etl_conn.begin() as conn:
    conn.execute(text(sql))

# load

In [6]:
dim_novedad_load = dim_novedad.set_index('key_dim_novedad')

dim_novedad_load.to_sql(
    "dim_novedad",
    con=etl_conn,
    if_exists="append",
    index=True,
    index_label='key_dim_novedad'
)

2